# Практика: бинарные деревья поиска

На лекции разобрано устройство бинарного дерева поиска (BST) и основные операции с ним. В этой практике вы будете работать с деревьями в коде: строить их, искать ключи и решать задачи, которые на деревьях поиска решаются быстро.

## Как устроена практика

- Класс узла, рисование дерева и стандартные операции уже написаны в ячейке «движок». Их нужно только вызывать.
- В заданиях **циклы и заголовки функций уже даны**. Вы пишете то, что внутри них: условия `if / elif / else`, переходы по дереву и ответ.
- Место для вашего кода отмечено строкой `# YOUR CODE HERE`.
- После каждого задания идёт проверка `assert`. Обычно она запускает ваш код на нескольких входных данных. Если всё верно, будет выведено `✅ Верно`. Если нет, появится сообщение, на каких данных код ошибся.
- Ячейки запускаются по порядку, клавишами **Shift + Enter**.

## Частые ошибки

| Сообщение | Причина | Что сделать |
|---|---|---|
| `AssertionError: ...` | Проверка не пройдена | Прочитать сообщение: в нём указано, на каких данных ошибка |
| `NameError: name '...' is not defined` | Имя не определено | Запустить ячейки выше, в первую очередь «движок»; проверить опечатки |
| `IndentationError` | Неверные отступы | Код внутри `if`, `while`, `def` сдвигается на 4 пробела относительно заголовка |
| `AttributeError: 'NoneType' object has no attribute ...` | Обращение к `.key`, `.left` или `.right` у пустого места (`None`) | Перед переходом проверять, что узел не `None` |
| `TypeError: '<' not supported ...` | Сравниваются несравнимые значения (например, число и `None`) | Проверить, что в переменной действительно ключ |

Если ноутбук ведёт себя странно, перезапустите ядро (**Kernel → Restart**) и выполните ячейки сверху заново.

## 0. Подготовка

Первая ячейка устанавливает библиотеку `matplotlib`, она нужна для рисунков. Если библиотека уже установлена, ячейка просто сообщит об этом.

Вторая ячейка — «движок»: класс узла и функции для работы с деревом. Разбирать её код не обязательно, но можно. Все функции в нём написаны циклами, без рекурсии.

In [ ]:
%pip install matplotlib

In [ ]:
# ======================= ДВИЖОК ПРАКТИКИ =======================
import math
import random
import matplotlib.pyplot as plt


class Node:
    """Узел дерева: ключ, значение (не обязательно) и ссылки на левого и правого потомка."""
    def __init__(self, key, value=None):
        self.key = key
        self.value = value
        self.left = None
        self.right = None

    def __repr__(self):
        return "Node(" + repr(self.key) + ")"


def insert(root, key, value=None):
    """Вставляет ключ и возвращает корень дерева."""
    new_node = Node(key, value)
    if root is None:
        return new_node
    node = root
    while True:
        if key == node.key:
            node.value = value
            return root
        if key < node.key:
            if node.left is None:
                node.left = new_node
                return root
            node = node.left
        else:
            if node.right is None:
                node.right = new_node
                return root
            node = node.right


def build_tree(keys):
    """Строит дерево, вставляя ключи из списка по очереди."""
    root = None
    for key in keys:
        root = insert(root, key)
    return root


def find(root, key):
    """Возвращает узел с ключом или None, если ключа нет."""
    node = root
    while node is not None:
        if key == node.key:
            return node
        node = node.left if key < node.key else node.right
    return None


def search_path(root, key):
    """Список ключей узлов, которые проверяет поиск."""
    path = []
    node = root
    while node is not None:
        path.append(node.key)
        if key == node.key:
            break
        node = node.left if key < node.key else node.right
    return path


def delete(root, key):
    """Удаляет ключ и возвращает корень дерева."""
    parent, node = None, root
    while node is not None and node.key != key:
        parent = node
        node = node.left if key < node.key else node.right
    if node is None:
        return root
    if node.left is not None and node.right is not None:
        heir_parent, heir = node, node.right
        while heir.left is not None:
            heir_parent, heir = heir, heir.left
        node.key, node.value = heir.key, heir.value
        parent, node = heir_parent, heir
    child = node.left if node.left is not None else node.right
    if parent is None:
        return child
    if parent.left is node:
        parent.left = child
    else:
        parent.right = child
    return root


def _all_nodes(root):
    nodes, stack = [], ([root] if root is not None else [])
    while stack:
        node = stack.pop()
        nodes.append(node)
        for child in (node.right, node.left):
            if child is not None:
                stack.append(child)
    return nodes


def count_nodes(root):
    """Количество узлов."""
    return len(_all_nodes(root))


def height(root):
    """Высота дерева (корень — уровень 0, у пустого дерева высота -1)."""
    if root is None:
        return -1
    level, h = [root], -1
    while level:
        h += 1
        level = [c for node in level for c in (node.left, node.right) if c is not None]
    return h


def inorder(root):
    """Ключи в порядке обхода: левое поддерево, узел, правое поддерево."""
    keys, stack, node = [], [], root
    while stack or node is not None:
        while node is not None:
            stack.append(node)
            node = node.left
        node = stack.pop()
        keys.append(node.key)
        node = node.right
    return keys


def leaves(root):
    """Ключи всех листьев по возрастанию."""
    return sorted(n.key for n in _all_nodes(root) if n.left is None and n.right is None)


def is_bst(root):
    """True, если правило дерева поиска выполняется для всех поддеревьев."""
    stack = [(root, None, None)]
    while stack:
        node, low, high = stack.pop()
        if node is None:
            continue
        if (low is not None and node.key <= low) or (high is not None and node.key >= high):
            return False
        stack.append((node.left, low, node.key))
        stack.append((node.right, node.key, high))
    return True


def parents_ok(root):
    """True, если каждый потомок правильно расположен относительно своего родителя."""
    for node in _all_nodes(root):
        if node.left is not None and not node.left.key < node.key:
            return False
        if node.right is not None and not node.right.key > node.key:
            return False
    return True


def same_tree(a, b):
    """True, если у двух деревьев одинаковая форма и одинаковые ключи на тех же местах."""
    stack = [(a, b)]
    while stack:
        x, y = stack.pop()
        if x is None or y is None:
            if x is not y:
                return False
            continue
        if x.key != y.key:
            return False
        stack.append((x.left, y.left))
        stack.append((x.right, y.right))
    return True


def average_steps(root, keys):
    """Среднее число узлов, которые проверяет поиск ключей из списка."""
    return sum(len(search_path(root, k)) for k in keys) / len(keys)


def broken_tree():
    """Дерево для задания 11: ключ одного узла изменён так, что правило нарушено."""
    tree = build_tree([50, 30, 70, 20, 40, 60, 80, 35, 45, 65, 10])
    find(tree, 45).key = 55
    return tree


def _layout(root):
    pos, stack, node, depth, i = {}, [], root, 0, 0
    while stack or node is not None:
        while node is not None:
            stack.append((node, depth))
            node, depth = node.left, depth + 1
        node, depth = stack.pop()
        pos[id(node)] = (i, depth)
        i += 1
        node, depth = node.right, depth + 1
    return pos


def draw_tree(root, path=None, highlight=None, title=None):
    """Рисует дерево. path — ключи пути (синие), highlight — выделенные ключи (оранжевые)."""
    if root is None:
        print("Дерево пустое.")
        return
    nodes = _all_nodes(root)
    if len(nodes) > 64:
        print("В дереве", len(nodes), "узлов, рисунок строится только для деревьев до 64 узлов.")
        return
    on_path = set(path or [])
    marked = set(highlight or [])
    pos = _layout(root)
    h = max(depth for _, depth in pos.values())
    wide = max(len(str(n.key)) for n in nodes) > 3
    width = min(16, max(4, len(nodes) * (1.25 if wide else 0.8) + 1))
    fig, ax = plt.subplots(figsize=(width, 1.1 * h + 1.8))
    for node in nodes:
        x, y = pos[id(node)]
        for child in (node.left, node.right):
            if child is not None:
                cx, cy = pos[id(child)]
                lit = node.key in on_path and child.key in on_path
                ax.plot([x, cx], [y, cy], color="#2456B8" if lit else "#1F2A44",
                        linewidth=4 if lit else 1.8, zorder=1)
    for node in nodes:
        x, y = pos[id(node)]
        if node.key in marked:
            face, text = "#F4A259", "#1F2A44"
        elif node.key in on_path:
            face, text = "#2456B8", "white"
        else:
            face, text = "#FFFDF8", "#1F2A44"
        label = str(node.key)
        if not wide:
            ax.scatter([x], [y], s=1300, c=face, edgecolors="#1F2A44", linewidths=1.8, zorder=2)
            ax.text(x, y, label, ha="center", va="center", fontsize=13,
                    fontweight="bold", color=text, zorder=3)
        else:
            ax.text(x, y, label, ha="center", va="center", fontsize=13,
                    fontweight="bold", color=text, zorder=3,
                    bbox=dict(boxstyle="round,pad=0.4", fc=face, ec="#1F2A44", lw=1.8))
    ax.set_xlim(-0.8, len(nodes) - 0.2)
    ax.set_ylim(h + 0.7, -0.7)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=14)
    plt.show()


def _balanced_order(low, high):
    order, ranges = [], [(low, high)]
    while ranges:
        a, b = ranges.pop(0)
        if a > b:
            continue
        mid = (a + b) // 2
        order.append(mid)
        ranges.append((a, mid - 1))
        ranges.append((mid + 1, b))
    return order


def play_guess_game(low=1, high=1000):
    """Программа угадывает загаданное число, спускаясь по сбалансированному дереву."""
    node = build_tree(_balanced_order(low, high))
    print("Загадайте целое число от", low, "до", str(high) + ".")
    print("Ответы: '=' — угадано, '<' — загаданное число меньше, '>' — больше.")
    questions = 0
    while node is not None:
        answer = input("Вопрос " + str(questions + 1) + ": это " + str(node.key) + "? ").strip()
        if answer not in ("=", "<", ">"):
            print("Введите один символ: =, < или >")
            continue
        questions += 1
        if answer == "=":
            print("Число найдено за", questions, "вопросов.")
            return
        node = node.left if answer == "<" else node.right
    print("Такого числа в диапазоне нет: в ответах есть противоречие.")


print("Движок загружен.")

### Функции движка

| Функция | Что делает |
|---|---|
| `build_tree(список)` | строит дерево, вставляя ключи по очереди; возвращает корень |
| `insert(tree, ключ)` / `insert(tree, ключ, значение)` | вставляет ключ (и значение); возвращает корень |
| `delete(tree, ключ)` | удаляет ключ; возвращает корень |
| `find(tree, ключ)` | узел с этим ключом или `None` |
| `search_path(tree, ключ)` | список ключей, которые проверяет поиск |
| `height(tree)`, `count_nodes(tree)` | высота дерева и число узлов |
| `inorder(tree)` | ключи при обходе «левое поддерево → узел → правое поддерево» |
| `leaves(tree)` | ключи листьев по возрастанию |
| `is_bst(tree)` | `True`, если это корректное дерево поиска |
| `draw_tree(tree, path=..., highlight=..., title=...)` | рисует дерево; `path` выделяет путь синим, `highlight` — узлы оранжевым |

## 1. Что понадобится из Python 

Условия `if / elif / else` вы уже знаете. Кроме них, в практике встретятся ещё несколько конструкций. Писать с нуля их не придётся, но читать и понимать нужно.

### 1.1 Узел и обращение через точку

Переменная `tree` хранит корень дерева. Поля узла:

- `node.key` — ключ;
- `node.left`, `node.right` — левый и правый потомок.

Обращения можно соединять в цепочку: `tree.left.right` — это правый потомок левого потомка корня. Если потомка нет, поле равно `None`. Проверяют это так: `node.left is None` (слева пусто) или `node.left is not None` (слева есть узел).

In [ ]:
tree = build_tree([8, 3, 10, 1, 6, 14, 4, 7, 13])
draw_tree(tree, title="Дерево с лекции")
print("Корень:", tree.key)
print("tree.left.right.key =", tree.left.right.key)
print("Левый потомок узла 1:", tree.left.left.left)
print("Слева от узла 1 пусто?", tree.left.left.left is None)

### 1.2 Цикл `while` и `break`

`while условие:` повторяет вложенные строки, пока условие истинно. `break` немедленно завершает цикл.

Пример ниже ищет минимальный ключ: от корня идём влево, пока слева есть узел.

In [ ]:
node = tree
while node.left is not None:
    node = node.left
print("Минимальный ключ:", node.key)

### 1.3 Функции

`def имя(параметры):` объявляет функцию: именованный блок кода, который можно вызывать с разными данными. `return значение` завершает функцию и возвращает результат.

В заданиях заголовок функции уже написан. Вы дописываете тело, а проверка вызывает функцию на нескольких наборах данных.

In [ ]:
def direction(x, node_key):
    if x < node_key:
        return "налево"
    elif x > node_key:
        return "направо"
    else:
        return "найдено"

print(direction(5, 8), direction(12, 8), direction(8, 8))

## 2. Структура дерева 

### Задание 1. Построение без компьютера

Ключи вставляются в пустое дерево в таком порядке:

**50, 30, 70, 20, 40, 60, 80, 35, 65, 45, 10**

Нарисуйте дерево на бумаге и заполните ответы. Рисунок появится только после того, как проверка пройдёт.

In [ ]:
# Задание 1
keys = [50, 30, 70, 20, 40, 60, 80, 35, 65, 45, 10]

parent_of_65 = 0      # ключ родителя узла 65
tree_height = 0       # высота дерева
leaf_keys = []        # ключи всех листьев, через запятую, в любом порядке
level_of_45 = 0       # уровень узла 45 (корень — уровень 0)
# YOUR CODE HERE

tree1 = build_tree(keys)
assert parent_of_65 == 60, "Неверный родитель узла 65. Пройдите путь вставки 65 от корня"
assert tree_height == height(tree1), "Неверная высота"
assert sorted(leaf_keys) == leaves(tree1), "Неверный набор листьев: лист — узел без потомков"
assert level_of_45 == len(search_path(tree1, 45)) - 1, "Неверный уровень узла 45"
draw_tree(tree1, title="Задание 1")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Каждый ключ начинает путь от корня. Например, 65: 65 > 50 → вправо к 70; 65 < 70 → влево к 60; 65 > 60 → вправо, там пусто.

Лист — узел, у которого нет ни левого, ни правого потомка.

</details>

In [ ]:
# Задание 2. Навигация по дереву с лекции (рисунок в разделе 1.1)
tree = build_tree([8, 3, 10, 1, 6, 14, 4, 7, 13])

node_13 = tree        # допишите цепочку из .left / .right, ведущую к узлу 13
node_4 = tree         # ... к узлу 4
key_lrr = 0           # чему равно tree.left.right.right.key? Ответьте, не запуская код
# YOUR CODE HERE

assert node_13 is not None and node_13.key == 13, "node_13 указывает не на узел 13"
assert node_4 is not None and node_4.key == 4, "node_4 указывает не на узел 4"
assert key_lrr == tree.left.right.right.key, "Неверное значение tree.left.right.right.key"
print("✅ Верно")

## 3. Поиск 

Правило поиска ключа `x`: начинаем с корня. Если `x` равен ключу узла — ключ найден. Если меньше — переходим в левое поддерево, если больше — в правое. Если дошли до `None`, ключа в дереве нет.

В функциях ниже есть переменная `guard`. Она ограничивает число шагов цикла, чтобы ошибка в коде не привела к бесконечному циклу. Её не нужно трогать.

In [ ]:
# Задание 3. Поиск с записью пути
# Функция возвращает список ключей всех проверенных узлов.
# Если ключ найден, он последний в списке. Если не найден, список заканчивается на узле, после которого начинается пустое место.

def search(tree, x):
    node = tree
    path = []
    guard = 0
    while node is not None and guard < 100:
        guard = guard + 1
        path.append(node.key)
        # YOUR CODE HERE
    return path


tree = build_tree([8, 3, 10, 1, 6, 14, 4, 7, 13])
for x in [13, 1, 7, 8, 9, 5, 20, 0]:
    assert search(tree, x) == search_path(tree, x), "x = " + str(x) + ": ожидался путь " + str(search_path(tree, x)) + ", получен " + str(search(tree, x)[:12])
draw_tree(tree, path=search(tree, 5), title="Путь поиска 5 (ключа нет)")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Внутри цикла нужны три ветки. В одной функция завершается и возвращает `path`, в двух других `node` переходит к одному из потомков. Если путь получается вида `[8, 8, 8, ...]`, значит, переменная `node` не меняется.

</details>

### Задание 4. Второе место в рейтинге

В дереве хранятся очки игроков. Максимум — самый правый узел. Нужно найти **второй по величине** ключ.

Подумайте, где он может находиться. Есть два принципиально разных случая. Они зависят от того, есть ли у максимального узла левое поддерево.

В функции уже написан спуск к максимуму: после цикла `node` — узел с максимальным ключом, `parent` — его родитель. Допишите остальное. Считайте, что в дереве не меньше двух узлов.

In [ ]:
# Задание 4
def second_best(tree):
    parent = None
    node = tree
    while node.right is not None:
        parent = node
        node = node.right
    # YOUR CODE HERE


tests = [
    [500, 320, 780, 150, 410, 650, 920, 270],
    [50, 30, 70, 60, 65],
    [5, 3],
    [10, 20, 15, 12],
    [1, 2, 3, 4],
]
for keys in tests:
    expected = sorted(keys)[-2]
    result = second_best(build_tree(keys))
    assert result == expected, "Дерево из " + str(keys) + ": ожидалось " + str(expected) + ", получено " + str(result)
draw_tree(build_tree(tests[1]), highlight=[65], title="Случай, когда второй по величине — не родитель максимума")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

- У максимума **нет** левого поддерева. Тогда второй по величине — его родитель.
- Левое поддерево **есть**. Тогда все его ключи меньше максимума, но больше всего остального. Второй по величине — максимум этого поддерева: перейдите в `node.left` и спуститесь вправо до упора (так же, как в готовом цикле).

</details>

### Задание 5. Скидка по сумме заказа

Магазин даёт скидку по порогам: от 500 ₽ — 1 %, от 1000 ₽ — 3 %, от 3000 ₽ — 5 %, от 5000 ₽ — 7 %, от 10 000 ₽ — 10 %, от 20 000 ₽ — 15 %. Пороги хранятся в дереве поиска: ключ — порог, значение — процент скидки.

Для суммы заказа нужно найти **наибольший порог, не превышающий сумму**. Перебирать все пороги нельзя: решение должно спускаться по дереву, как поиск.

Рассуждайте так. Узел с порогом `node.key` либо подходит (`node.key <= amount`), либо нет.
- Если подходит, это кандидат в ответ. Где тогда могут найтись подходящие пороги **больше** него?
- Если не подходит, где могут быть подходящие пороги?

In [ ]:
# Задание 5
def discount_for(tree, amount):
    node = tree
    best = None            # лучший подходящий узел из найденных
    guard = 0
    while node is not None and guard < 100:
        guard = guard + 1
        # YOUR CODE HERE
    if best is None:
        return 0
    return best.value


discounts = None
discounts = insert(discounts, 5000, 7)
discounts = insert(discounts, 1000, 3)
discounts = insert(discounts, 20000, 15)
discounts = insert(discounts, 500, 1)
discounts = insert(discounts, 3000, 5)
discounts = insert(discounts, 10000, 10)
draw_tree(discounts, title="Пороги скидок")

cases = {400: 0, 500: 1, 999: 1, 2999: 3, 3000: 5, 7500: 7, 10000: 10, 19999: 10, 1000000: 15}
for amount in cases:
    result = discount_for(discounts, amount)
    assert result == cases[amount], "Сумма " + str(amount) + ": ожидалась скидка " + str(cases[amount]) + " %, получено " + str(result)
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Если порог подходит, запомните его в `best`. Большие подходящие пороги могут быть только **справа**. Если не подходит, подходящие пороги могут быть только **слева**.

</details>

## 4. Вставка 

### Задание 6. Вставка без готовой функции

Напишите вставку самостоятельно. Новый узел `new_node` уже создан. Спускайтесь от корня, как при поиске. Когда в нужном направлении окажется пусто, прикрепите туда `new_node` (например, `node.left = new_node`) и завершите цикл командой `break`.

Считайте, что дерево не пустое, а ключа `x` в нём ещё нет. Проверка строит дерево вашей функцией и сравнивает его с деревом, построенным функцией из движка: форма должна совпасть полностью.

In [ ]:
# Задание 6
def my_insert(tree, x):
    new_node = Node(x)
    node = tree
    guard = 0
    while guard < 100:
        guard = guard + 1
        # YOUR CODE HERE
    return tree


for keys in [[50, 30, 70, 20, 40, 60, 80, 35, 65], [1, 2, 3, 4, 5], [5, 4, 3, 2, 1], [8, 3, 10, 1, 6, 14, 4, 7, 13]]:
    mine = Node(keys[0])
    for k in keys[1:]:
        mine = my_insert(mine, k)
    assert count_nodes(mine) == len(keys), "Ключи " + str(keys) + ": в дереве " + str(count_nodes(mine)) + " узлов вместо " + str(len(keys))
    assert same_tree(mine, build_tree(keys)), "Ключи " + str(keys) + ": форма дерева не совпадает с правильной"
draw_tree(mine, title="Дерево, построенное my_insert")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Понадобятся вложенные условия. Внешнее решает направление (`x < node.key` или нет). Внутреннее проверяет, пусто ли в этом направлении. Если пусто — прикрепляем узел и `break`. Если нет — переходим к потомку.

</details>

## 5. Общий предок 

### Задание 7

**Наименьший общий предок** узлов `a` и `b` — самый нижний узел, у которого оба узла лежат в поддереве (сам узел тоже считается своим поддеревом). Это место, где пути поиска `a` и `b` от корня расходятся.

Такая задача возникает, например, в иерархии подразделений: найти ближайший общий отдел двух сотрудников.

Спускайтесь от корня и сравнивайте `a` и `b` с ключом текущего узла. Сколько случаев нужно различать и что делать в каждом?

In [ ]:
# Задание 7
def common_ancestor(tree, a, b):
    node = tree
    guard = 0
    while node is not None and guard < 100:
        guard = guard + 1
        # YOUR CODE HERE
    return None


tree = build_tree([50, 30, 70, 20, 40, 60, 80, 35, 65, 45, 10])
cases = [(35, 45, 40), (10, 45, 30), (65, 80, 70), (35, 65, 50), (20, 10, 20), (60, 65, 60), (45, 45, 45)]
for a, b, expected in cases:
    result = common_ancestor(tree, a, b)
    assert result == expected, "a = " + str(a) + ", b = " + str(b) + ": ожидалось " + str(expected) + ", получено " + str(result)
draw_tree(tree, highlight=[35, 45], path=[50, 30, 40], title="Общий предок 35 и 45 — узел 40")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Если `a` и `b` оба меньше ключа узла, оба лежат слева, и ответ тоже слева. Если оба больше — справа. Во всех остальных случаях (они по разные стороны или один из них равен ключу) текущий узел и есть ответ.

Условие «оба меньше» записывается так: `a < node.key and b < node.key`.

</details>

## 6. Форма дерева и число шагов 

Число шагов поиска не больше `высота + 1`. Высота же зависит от порядка вставки: одни и те же ключи могут дать и сбалансированное дерево, и «палку».

In [ ]:
stick = build_tree([1, 2, 3, 4, 5, 6, 7])
balanced = build_tree([4, 2, 6, 1, 3, 5, 7])
draw_tree(stick, title="Порядок 1..7: высота " + str(height(stick)))
draw_tree(balanced, title="Порядок 4, 2, 6, 1, 3, 5, 7: высота " + str(height(balanced)))

In [ ]:
# Задание 8. Порядок вставки
# а) Расположите числа 1..15 так, чтобы высота дерева была 3 (минимально возможная).
order_a = list(range(1, 16))
# б) Расположите числа 1..7 так, чтобы получилась «палка» (высота 6),
#    но порядок не был ни возрастающим, ни убывающим.
order_b = [1, 2, 3, 4, 5, 6, 7]
# YOUR CODE HERE

assert sorted(order_a) == list(range(1, 16)), "а) нужны числа 1..15, каждое по одному разу"
assert height(build_tree(order_a)) == 3, "а) высота получилась " + str(height(build_tree(order_a))) + ", нужна 3"
assert sorted(order_b) == list(range(1, 8)), "б) нужны числа 1..7, каждое по одному разу"
assert order_b != list(range(1, 8)) and order_b != list(range(7, 0, -1)), "б) порядок не должен быть монотонным"
assert height(build_tree(order_b)) == 6, "б) высота получилась " + str(height(build_tree(order_b))) + ", нужна 6"
draw_tree(build_tree(order_a), title="а) " + str(order_a))
draw_tree(build_tree(order_b), title="б) " + str(order_b))
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

а) Корень должен делить ключи пополам. Затем то же для каждой половины.

б) У каждого узла должен быть ровно один потомок. Значит, каждый следующий ключ должен быть наименьшим или наибольшим из ещё не вставленных.

</details>

### Задание 9. Сколько нужно вопросов

На каждом уровне сбалансированного дерева помещается вдвое больше узлов, чем на предыдущем: 1, 2, 4, 8, ... Поэтому дерево высоты `h` вмещает не более `2^(h+1) − 1` ключей.

Игра: загадано число от 1 до N. Можно спрашивать «это X?» и получать ответ «да / меньше / больше». Каждый вопрос — один шаг по сбалансированному дереву поиска.

Определите **без компьютера**, сколько вопросов понадобится в худшем случае при оптимальной стратегии для N = 100, N = 1000 и N = 1 000 000.

In [ ]:
# Задание 9
questions_100 = 0
questions_1000 = 0
questions_million = 0
# YOUR CODE HERE

for n, answer in [(100, questions_100), (1000, questions_1000), (1000000, questions_million)]:
    assert answer == math.floor(math.log2(n)) + 1, "N = " + str(n) + ": неверно. Найдите наименьшее число уровней, вмещающее N ключей"
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Если уровней `k`, в дерево помещается `2^k − 1` ключей. Для N = 100: при 6 уровнях помещается 63 ключа — мало, при 7 уровнях — 127, достаточно.

</details>

Проверьте результат вживую: программа ниже угадывает число от 1 до 1000, спускаясь по сбалансированному дереву. Ответы вводятся в поле ввода.

In [ ]:
play_guess_game(1, 1000)

На практике ключи редко приходят в удачном порядке. Эксперимент ниже строит дерево из 100 000 случайных ключей и сравнивает его высоту с минимально возможной. Выполнение займёт несколько секунд.

In [ ]:
random.seed(2026)
orders = random.sample(range(1, 10_000_000), 100_000)
big_tree = build_tree(orders)

print("Узлов:", count_nodes(big_tree))
print("Высота случайного дерева:", height(big_tree))
print("Минимально возможная высота:", math.floor(math.log2(100_000)))
print("Среднее число шагов поиска:", round(average_steps(big_tree, orders[:5000]), 1))
print("Линейный поиск по списку в худшем случае:", len(orders), "шагов")

## 7. Удаление 

Три случая удаления узла:

1. Нет потомков — узел просто убирается.
2. Один потомок — потомок занимает место узла.
3. Два потомка — ключ узла заменяется наименьшим ключом правого поддерева (самым левым узлом правого поддерева). Затем этот узел удаляется по случаю 1 или 2.

In [ ]:
# Задание 10. Дерево из задания 1. Ответьте, не запуская удаление.
keys = [50, 30, 70, 20, 40, 60, 80, 35, 65, 45, 10]

key_in_place_of_30 = 0     # какой ключ окажется на месте 30 после delete(tree, 30)?
root_after_50 = 0          # затем удаляем 50: какой ключ станет корнем?
parent_of_65_after = 0     # кто после этого будет родителем узла 65?
# YOUR CODE HERE

tree = build_tree(keys)
tree = delete(tree, 30)
assert tree.left.key == key_in_place_of_30, "Неверно: на месте 30 стоит другой ключ"
tree = delete(tree, 50)
assert tree.key == root_after_50, "Неверный новый корень"
assert parent_of_65_after == search_path(tree, 65)[-2], "Неверный родитель узла 65"
draw_tree(tree, highlight=[key_in_place_of_30, root_after_50], title="После удаления 30 и 50")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Правое поддерево узла 30 начинается с 40; самый левый узел в нём — 35.

Правое поддерево 50 начинается с 70; самый левый узел — 60. Когда 60 уходит в корень, его старое место освобождается. Что происходит с его единственным потомком 65?

</details>

## 8. Поиск ошибки 

### Задание 11

В дереве ниже ключ одного узла испорчен. Каждая пара «родитель — потомок» выглядит правильно, но дерево не является деревом поиска.

1. Найдите ошибочный ключ по рисунку.
2. Проверьте: `find(tree, ошибочный_ключ)` вернёт `None`, хотя узел в дереве есть. Объясните, почему так происходит.
3. Исправьте ключ: замените его на любое значение, при котором дерево станет корректным. Раз `find` здесь не поможет, дойдите до узла по цепочке, например: `tree.left.key = новый_ключ`.

In [ ]:
# Задание 11
tree = broken_tree()
draw_tree(tree, title="В дереве есть ошибка")

wrong_key = 0      # ошибочный ключ
print("find(tree, wrong_key):", find(tree, wrong_key))
# YOUR CODE HERE   (исправление ключа)

assert wrong_key == 55, "Это не тот узел. Проверьте каждый узел относительно ВСЕХ его предков, а не только родителя"
assert 55 not in inorder(tree), "Ключ 55 всё ещё в дереве — исправьте его"
assert is_bst(tree), "Дерево всё ещё некорректно: новое значение должно подходить ко всем предкам узла"
assert count_nodes(tree) == 11, "Число узлов изменилось — нужно только изменить один ключ"
draw_tree(tree, title="Исправленное дерево")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Узел с ключом 55 — правый потомок 40, и относительно 40 это верно. Но он лежит в **левом** поддереве корня 50. Поиск 55 в корне сравнивает 55 > 50 и уходит **вправо**, поэтому до этого узла никогда не доходит.

Путь к узлу: корень → влево → вправо → вправо. Подходит значение больше 40 и меньше 50, которого ещё нет в дереве.

</details>

## 9. Итоговая задача: телефонная книга 
Контакты хранятся в дереве поиска: ключ — имя, значение — номер телефона. Строки сравниваются по алфавиту: `"Анна" < "Борис"` истинно. Поэтому всё, что работало с числами, работает и с именами.

Исходные данные заданы **словарём** — набором пар «ключ: значение». `contacts["Маша"]` возвращает номер Маши, а `for name in contacts:` перебирает все имена.

In [ ]:
# Задание 12, часть 1. Постройте книгу: вставьте в дерево каждое имя с его номером.
contacts = {
    "Маша": "+7 900 111-11-11",
    "Денис": "+7 900 222-22-22",
    "Света": "+7 900 333-33-33",
    "Артём": "+7 900 444-44-44",
    "Ира": "+7 900 555-55-55",
    "Петя": "+7 900 666-66-66",
    "Яна": "+7 900 777-77-77",
    "Борис": "+7 900 888-88-88",
    "Катя": "+7 900 999-99-99",
    "Олег": "+7 900 000-00-00",
}
book = None
for name in contacts:
    # YOUR CODE HERE
    pass

assert count_nodes(book) == 10, "В книге должно быть 10 контактов, сейчас " + str(count_nodes(book))
assert find(book, "Олег").value == "+7 900 000-00-00", "Номера должны сохраниться как значения узлов"
assert inorder(book) == sorted(contacts), "Нарушен порядок дерева"
draw_tree(book, title="Телефонная книга")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Одна строка внутри цикла: `insert` с тремя аргументами (дерево, имя, номер), результат присваивается обратно в `book`.

</details>

In [ ]:
# Задание 12, часть 2. Номер по имени.
# Функция возвращает номер телефона, а если имени нет в книге — строку "не найден".

def phone_of(book, name):
    node = find(book, name)
    # YOUR CODE HERE


assert phone_of(book, "Ира") == "+7 900 555-55-55", "Неверный номер для существующего имени"
assert phone_of(book, "Яна") == "+7 900 777-77-77", "Неверный номер для существующего имени"
assert phone_of(book, "Гоша") == "не найден", "Для отсутствующего имени должно возвращаться \"не найден\""
print("✅ Верно")

### Задание 12, часть 3. Автодополнение

Пользователь набирает начало имени, например `"Ка"`. Программа должна предложить **первое по алфавиту имя, которое не меньше введённой строки**: для `"Ка"` это `"Катя"`. Если такого имени нет, возвращается `None`.

Задача зеркальна заданию 5: там искался наибольший ключ не больше суммы, здесь — наименьший ключ не меньше строки. Решение должно спускаться по дереву, а не перебирать все имена.

In [ ]:
# Задание 12, часть 3
def suggest(book, prefix):
    node = book
    best = None
    guard = 0
    while node is not None and guard < 100:
        guard = guard + 1
        # YOUR CODE HERE
    if best is None:
        return None
    return best.key


cases = {"Ка": "Катя", "Б": "Борис", "Ж": "Ира", "А": "Артём", "Ол": "Олег", "Пе": "Петя", "Я": "Яна", "Яр": None}
for prefix in cases:
    result = suggest(book, prefix)
    assert result == cases[prefix], "Ввод \"" + prefix + "\": ожидалось " + str(cases[prefix]) + ", получено " + str(result)
draw_tree(book, path=search_path(book, "Ка"), highlight=["Катя"], title="Подсказка для \"Ка\"")
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Если `node.key >= prefix`, имя подходит и становится кандидатом. Меньшие подходящие имена могут быть только **слева**. Если не подходит, искать нужно **справа**.

</details>

## Дополнительное задание: ближайший ключ

Найдите в дереве ключ, ближайший к числу `x`, то есть с наименьшим `abs(x - key)`. Функция `abs` возвращает модуль числа.

Решение должно спускаться по дереву за один проход. На каждом шаге сравните текущий узел с лучшим найденным, а затем выберите направление так же, как при поиске.

In [ ]:
# Дополнительное задание
def nearest(tree, x):
    node = tree
    best = tree.key
    guard = 0
    while node is not None and guard < 100:
        guard = guard + 1
        # YOUR CODE HERE
    return best


tree = build_tree([8, 3, 10, 1, 6, 14, 4, 7, 13])
cases = {12: 13, 0: 1, 100: 14, 5.4: 6, 11.6: 13, 7: 7, 9.2: 10, 2.6: 3}
for x in cases:
    result = nearest(tree, x)
    assert result == cases[x], "x = " + str(x) + ": ожидалось " + str(cases[x]) + ", получено " + str(result)
print("✅ Верно")

<details>
<summary><b>Подсказка</b></summary>

Два независимых действия на каждом шаге:
1. если `abs(x - node.key) < abs(x - best)`, обновить `best`;
2. перейти влево или вправо (или завершить цикл, если `x == node.key`).

</details>

## Итоги

- В дереве поиска каждое сравнение отсекает одно из поддеревьев. Поэтому поиск, вставка, поиск порога и общего предка выполняются за число шагов, не большее `высота + 1`.
- Одна и та же схема спуска по дереву (сравнить, выбрать направление) решает разные задачи. Меняется только то, что запоминается по пути.
- Высота зависит от порядка вставки: от `log₂ N` для сбалансированного дерева до `N − 1` для «палки».